# LAB 04: FEATURE SELECTION VÀ ĐÁNH GIÁ HIỆU SUẤT MÔ HÌNH (BẢN FULL)


In [ ]:
import pandas as pd
import numpy as np
import time
import os
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_regression, RFE, SelectFromModel, mutual_info_regression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import warnings
warnings.filterwarnings('ignore')

os.makedirs('data', exist_ok=True)
os.makedirs('figures', exist_ok=True)



## 1. Đọc và Tiền xử lý dữ liệu


In [ ]:
# Đọc dữ liệu
df = pd.read_csv('data/train.csv')

if 'Id' in df.columns:
    df = df.drop('Id', axis=1)

X = df.drop('SalePrice', axis=1)
y = df['SalePrice']

numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = X.select_dtypes(include=['object']).columns

# Xử lý missing value
num_imputer = SimpleImputer(strategy='median')
X_num = pd.DataFrame(num_imputer.fit_transform(X[numeric_cols]), columns=numeric_cols)

cat_imputer = SimpleImputer(strategy='constant', fill_value='None')
X_cat = pd.DataFrame(cat_imputer.fit_transform(X[categorical_cols]), columns=categorical_cols)

# Label Encoding
for col in categorical_cols:
    le = LabelEncoder()
    X_cat[col] = le.fit_transform(X_cat[col])

X_processed = pd.concat([X_num, X_cat], axis=1)

X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_processed.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_processed.columns)



## 2. Rút trích đặc trưng (Feature Selection)


In [ ]:
k_features = 20

# 1. Filter Method: F-test
filter_f = SelectKBest(score_func=f_regression, k=k_features)
X_train_f = filter_f.fit_transform(X_train_scaled, y_train)
X_test_f = filter_f.transform(X_test_scaled)

# 2. Filter Method: Mutual Info
filter_mi = SelectKBest(score_func=mutual_info_regression, k=k_features)
X_train_mi = filter_mi.fit_transform(X_train_scaled, y_train)
X_test_mi = filter_mi.transform(X_test_scaled)

# 3. Wrapper Method: RFE
rfe_selector = RFE(estimator=RandomForestRegressor(n_estimators=20, random_state=42), n_features_to_select=k_features)
X_train_rfe = rfe_selector.fit_transform(X_train_scaled, y_train)
X_test_rfe = rfe_selector.transform(X_test_scaled)

# 4. Embedded Method: XGBoost Feature Importance
embed_selector = SelectFromModel(XGBRegressor(random_state=42), max_features=k_features)
X_train_embed = embed_selector.fit_transform(X_train_scaled, y_train)
X_test_embed = embed_selector.transform(X_test_scaled)

datasets = {
    "All Features (Baseline)": (X_train_scaled, X_test_scaled),
    "Filter F-test": (X_train_f, X_test_f),
    "Filter Mutual Info": (X_train_mi, X_test_mi),
    "Wrapper RFE": (X_train_rfe, X_test_rfe),
    "Embedded XGBoost FI": (X_train_embed, X_test_embed)
}



## 3. Huấn luyện mô hình


In [ ]:
models = {
    "SVM": SVR(kernel='linear'),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "XGBoost": XGBRegressor(objective='reg:squarederror', random_state=42)
}

results = []

for data_name, (X_tr, X_te) in datasets.items():
    for model_name, model in models.items():
        start_time = time.time()
        model.fit(X_tr, y_train)
        train_time = time.time() - start_time
        
        y_pred = model.predict(X_te)
        
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2 = r2_score(y_test, y_pred)
        
        results.append({
            "Dataset": data_name,
            "Model": model_name,
            "RMSE": rmse,
            "R2_Score": r2,
            "Train_Time(s)": round(train_time, 4)
        })

results_df = pd.DataFrame(results)
display(results_df)



## 4. TRỰC QUAN HÓA DỮ LIỆU ĐẦY ĐỦ (8 BIỂU ĐỒ)


In [ ]:
sns.set_theme(style="whitegrid")

# --- Fig 1: Heatmap R2 Score ---
plt.figure(figsize=(12, 4))
pivot_r2 = results_df.pivot(index="Model", columns="Dataset", values="R2_Score")
sns.heatmap(pivot_r2, annot=True, cmap="YlGnBu", fmt=".4f", linewidths=.5)
plt.title("R² Score – Phương pháp Feature Selection × Mô hình", fontweight='bold', pad=15)
plt.ylabel("Mô hình")
plt.xlabel("Phương pháp Feature Selection")
plt.tight_layout()
plt.savefig('figures/fig1_r2_heatmap.png', dpi=300)
plt.show()

# --- Fig 2: Barplot RMSE ---
plt.figure(figsize=(12, 5))
sns.barplot(data=results_df, x='Dataset', y='RMSE', hue='Model', palette='husl')
plt.title("RMSE theo phương pháp Feature Selection và mô hình", fontweight='bold')
plt.ylabel("RMSE (VND)")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('figures/fig2_rmse_bar.png', dpi=300)
plt.show()

# --- Fig 3: Feature Importance (XGBoost Embedded) ---
xgb_full = XGBRegressor(random_state=42).fit(X_train_scaled, y_train)
fi = pd.Series(xgb_full.feature_importances_, index=X_processed.columns).sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 6))
sns.barplot(x=fi.values, y=fi.index, palette="YlOrRd_r")
plt.title("Top 15 Feature quan trọng nhất (Embedded – XGBoost)", fontweight='bold')
plt.xlabel("Feature Importance (XGBoost)")
plt.tight_layout()
plt.savefig('figures/fig3_feature_importance.png', dpi=300)
plt.show()

# --- Fig 4: Lineplot R2 Score ---
plt.figure(figsize=(12, 5))
sns.lineplot(data=results_df, x='Dataset', y='R2_Score', hue='Model', marker='o', linewidth=2, markersize=8)
plt.title("So sánh R² giữa các phương pháp Feature Selection", fontweight='bold')
plt.ylim(0, 1.05)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('figures/fig4_r2_lineplot.png', dpi=300)
plt.show()

# --- Fig 5: Correlation Heatmap ---
top_features = fi.index.tolist()
corr_df = pd.concat([X_processed[top_features], y], axis=1)
corr_matrix = corr_df.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, mask=mask, cmap='vlag', annot=True, fmt=".2f", 
            square=True, linewidths=.5, cbar_kws={"shrink": .8})
plt.title("Ma trận tương quan – Top 15 Features + SalePrice", fontweight='bold')
plt.tight_layout()
plt.savefig('figures/fig5_correlation.png', dpi=300)
plt.show()

# --- Fig 6: Phân phối SalePrice ---
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.histplot(y, kde=True, ax=axes[0], color="dodgerblue")
axes[0].set_title("Phân phối SalePrice gốc")
axes[0].set_xlabel("Giá nhà (USD)")

sns.histplot(np.log1p(y), kde=True, ax=axes[1], color="forestgreen")
axes[1].set_title("Phân phối log(SalePrice)")
axes[1].set_xlabel("log(Giá nhà)")
plt.tight_layout()
plt.savefig('figures/fig6_price_dist.png', dpi=300)
plt.show()

# --- Fig 7 & 8: Đã có từ file cũ (Barplot R2 và Train Time) ---
plt.figure(figsize=(10, 5))
sns.barplot(data=results_df, x='Model', y='R2_Score', hue='Dataset', palette="viridis")
plt.title('So Sánh R2 Score Giữa Các Mô Hình và Feature Selection', fontweight='bold')
plt.ylim(0, 1.1)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('figures/r2_score_comparison.png', dpi=300)
plt.close()

plt.figure(figsize=(10, 5))
sns.barplot(data=results_df, x='Model', y='Train_Time(s)', hue='Dataset', palette="magma")
plt.title('So Sánh Thời Gian Huấn Luyện (Train Time)', fontweight='bold')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('figures/train_time_comparison.png', dpi=300)
plt.close()

print("ĐÃ LƯU TOÀN BỘ 8 BIỂU ĐỒ VÀO THƯ MỤC 'figures/'")

